In [1]:
# ============================================================
# 03_feature_engineering.ipynb — Phase 3: Feature Engineering
# ============================================================

import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

os.makedirs('../models', exist_ok=True)

df = pd.read_csv('../data/processed/cleaned_data.csv')
print(df.shape)
df.head()
# ============================
# Handle missing values
# ============================

# Fill diagnosis codes with "Missing"
for col in ['diag_1', 'diag_2', 'diag_3']:
    if col in df.columns:
        df[col] = df[col].fillna("Missing")

# Fill glucose and A1C categorical values with "None"
for col in ['max_glu_serum', 'A1Cresult']:
    if col in df.columns:
        df[col] = df[col].fillna("None")


(69973, 52)


In [2]:
df['total_prior_utilization'] = (
    df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']
)

print(df[['number_outpatient', 'number_emergency', 'number_inpatient',
          'total_prior_utilization']].describe())

       number_outpatient  number_emergency  number_inpatient  \
count       69973.000000      69973.000000      69973.000000   
mean            0.279536          0.103912          0.176254   
std             1.064035          0.511870          0.601657   
min             0.000000          0.000000          0.000000   
25%             0.000000          0.000000          0.000000   
50%             0.000000          0.000000          0.000000   
75%             0.000000          0.000000          0.000000   
max            42.000000         42.000000         12.000000   

       total_prior_utilization  
count             69973.000000  
mean                  0.559702  
std                   1.427760  
min                   0.000000  
25%                   0.000000  
50%                   0.000000  
75%                   1.000000  
max                  49.000000  


In [3]:
med_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
            'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
            'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
            'miglitol', 'troglitazone', 'tolazamide', 'examide',
            'citoglipton', 'insulin', 'glyburide-metformin',
            'glipizide-metformin', 'glimepiride-pioglitazone',
            'metformin-rosiglitazone', 'metformin-pioglitazone']

# keep only columns that actually exist in your cleaned df
med_cols = [c for c in med_cols if c in df.columns]

def count_changes(row):
    return sum(1 for c in med_cols if row[c] in ['Up', 'Down'])

df['num_meds_changed'] = df.apply(count_changes, axis=1)

print(df['num_meds_changed'].value_counts().sort_index())
print(df['num_meds_changed'].describe())

num_meds_changed
0    52725
1    16252
2      919
3       74
4        3
Name: count, dtype: int64
count    69973.000000
mean         0.261872
std          0.475842
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          4.000000
Name: num_meds_changed, dtype: float64


In [4]:
# Define target and drop leakage-prone / ID columns
target_col = 'readmitted_binary'

drop_cols = ['patient_nbr', 'readmitted', target_col]  # drop raw target variants + ID
drop_cols = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=drop_cols)
y = df[target_col]

# Identify column types
numeric_features = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                     'num_medications', 'number_diagnoses', 'total_prior_utilization',
                     'num_meds_changed']
numeric_features = [c for c in numeric_features if c in X.columns]

nominal_features = ['race', 'gender', 'admission_type_id', 'discharge_disposition_id',
                     'admission_source_id', 'diag_1_group', 'diag_2_group', 'diag_3_group',
                     'max_glu_serum', 'A1Cresult', 'change', 'diabetesMed']
nominal_features = [c for c in nominal_features if c in X.columns]

ordinal_features = ['age_ordinal']
ordinal_features = [c for c in ordinal_features if c in X.columns]

high_card_feature = ['medical_specialty']
high_card_feature = [c for c in high_card_feature if c in X.columns]

print("Numeric:", numeric_features)
print("Nominal:", nominal_features)
print("Ordinal:", ordinal_features)
print("High-cardinality:", high_card_feature)

Numeric: ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_diagnoses', 'total_prior_utilization', 'num_meds_changed']
Nominal: ['race', 'gender', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'diag_1_group', 'diag_2_group', 'diag_3_group', 'max_glu_serum', 'A1Cresult', 'change', 'diabetesMed']
Ordinal: ['age_ordinal']
High-cardinality: ['medical_specialty']


In [5]:
# ColumnTransformer: different treatment per column type
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_features),
        ('ord', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ordinal_features),
        ('spec', OneHotEncoder(handle_unknown='ignore', sparse_output=False,
                                max_categories=15), high_card_feature),
    ],
    remainder='drop'
)

pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

In [6]:
# Fit on full X to build the transformer (actual model fitting happens later in Phase 4
# on X_train only — this just builds/tests the transformer shape)
pipeline.fit(X)

X_transformed = pipeline.transform(X)
print("Transformed feature matrix shape:", X_transformed.shape)

# Save the fitted pipeline
joblib.dump(pipeline, '../models/preprocessing_pipeline.joblib')
print("Saved: models/preprocessing_pipeline.joblib")

Transformed feature matrix shape: (69973, 120)
Saved: models/preprocessing_pipeline.joblib


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("Train target balance:\n", y_train.value_counts(normalize=True))
print("Test target balance:\n", y_test.value_counts(normalize=True))

# Save splits for Phase 4 (so clustering/classification notebooks can load directly)
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Splits saved to data/processed/")

X_train: (55978, 51)  X_test: (13995, 51)
Train target balance:
 readmitted_binary
0    0.910286
1    0.089714
Name: proportion, dtype: float64
Test target balance:
 readmitted_binary
0    0.910325
1    0.089675
Name: proportion, dtype: float64
Splits saved to data/processed/


In [8]:
import numpy as np
X_train_transformed = pipeline.transform(X_train)
print("Any NaNs in transformed data?", np.isnan(X_train_transformed).any())

Any NaNs in transformed data? False


In [9]:
print([c for c in X_train.columns if 'readmit' in c.lower()])

[]
